# qwen-front
Qwen-Image-Edit-2509 (GGUF Q4) frontalisation of 50 portraits, same instruction as the FLUX Kontext run. Output: /kaggle/working/qwen_front.

In [ ]:
import os, glob, shutil
BASE = '/kaggle/temp'
os.makedirs(f'{BASE}/code', exist_ok=True)
os.environ['HF_HOME'] = f'{BASE}/hf_cache'
tok = None
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as e:
    print('no Kaggle secret HF_TOKEN:', type(e).__name__)
if not tok:
    for f in glob.glob('/kaggle/input/**/hf_token.txt', recursive=True):
        tok = open(f).read().strip()
if tok:
    os.environ['HF_TOKEN'] = tok
import torch
assert torch.cuda.is_available(), 'ERROR: CUDA GPU is required.'
print(torch.cuda.get_device_name(0), torch.cuda.device_count(), 'GPU(s), torch', torch.__version__)
print('free disk /kaggle/temp (GB):', shutil.disk_usage(BASE).free // 2**30)

In [ ]:
!pip install -q -U diffusers transformers accelerate bitsandbytes gguf sentencepiece protobuf peft
!pip uninstall -y -q torchao
!python -c "from diffusers import QwenImageEditPlusPipeline, GGUFQuantizationConfig; print('ok')"

In [ ]:
open('/kaggle/temp/code/qwen_front.py', 'w', encoding='utf-8').write('"""Frontalise portraits with Qwen-Image-Edit-2509 on a 16GB T4, to compare against FLUX Kontext (real2pixar_front.py, same instruction).\n\n  python qwen_front.py --input <dir of <id>.png> --output <out dir> [--limit N] [--gguf Qwen-Image-Edit-2509-Q4_K_M.gguf] [--dtype fp16|bf16]\nMemory recipe (same idea as the FLUX scripts): the VL text encoder (NF4) embeds every prompt+image first and is dropped, then the transformer is\nloaded as a GGUF (QuantStack/Qwen-Image-Edit-2509-GGUF). Writes <out>/<id>.png (resumable) and <out>/params.json.\n"""\nimport argparse, gc, json, time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom PIL import Image\n\nMODEL = "Qwen/Qwen-Image-Edit-2509"\nGGUF_REPO = "QuantStack/Qwen-Image-Edit-2509-GGUF"\nFRONT = ("Rotate the head so the person faces the camera directly: a straight front-facing portrait, head upright, both eyes looking at the camera, "\n         "symmetric pose. Keep the same person: identity, face shape, hair, skin tone, expression, clothing and background.")  # same text as the Kontext run\nNEG = " "\n\nap = argparse.ArgumentParser()\nap.add_argument("--input"); ap.add_argument("--output"); ap.add_argument("--limit", type=int, default=0)\nap.add_argument("--gguf", default="Qwen-Image-Edit-2509-Q4_K_M.gguf"); ap.add_argument("--dtype", default="fp16", choices=["fp16", "bf16"])\nap.add_argument("--steps", type=int, default=20); ap.add_argument("--cfg", type=float, default=4.0)\nap.add_argument("--res", type=int, default=512); ap.add_argument("--seed", type=int, default=42)\nap.add_argument("--shard", default="0/1", help="k/n: process every n-th image starting at k (one process per GPU)")\nargs = ap.parse_args()\nassert torch.cuda.is_available(), "CUDA GPU required"\ndev = "cuda"; dtype = torch.float16 if args.dtype == "fp16" else torch.bfloat16\nR = args.res\nout = Path(args.output); out.mkdir(parents=True, exist_ok=True)\nfiles = sorted(Path(args.input).glob("*.png"))\nfiles = files[: args.limit] if args.limit else files\nshard_k, shard_n = map(int, args.shard.split("/"))\nfiles = files[shard_k::shard_n]\nassert files, f"no png in {args.input}"\ntodo = [p for p in files if not (out / p.name).exists()]\nif not todo:\n    print("nothing to do"); raise SystemExit(0)\n\n\ndef free():\n    gc.collect(); torch.cuda.empty_cache()\n\n\nimport diffusers.pipelines.qwenimage.pipeline_qwenimage_edit_plus as m\nfrom diffusers import GGUFQuantizationConfig, QwenImageEditPlusPipeline, QwenImageTransformer2DModel\nfrom huggingface_hub import hf_hub_download\nfrom transformers import BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration\n\nm.VAE_IMAGE_SIZE = R * R  # conditioning latents at the output size instead of 1024^2 (far fewer tokens on a T4)\nimport fcntl  # shards load one at a time: two 10GB GGUF + text-encoder loads at once would exhaust the 29GB of host RAM\nload_lock = open("/tmp/qwen_load.lock", "w"); fcntl.flock(load_lock, fcntl.LOCK_EX)\n\n# 1) encode prompt+image with the NF4 text encoder, keep the embeddings on the CPU, drop the encoder\nte = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL, subfolder="text_encoder", torch_dtype=dtype, device_map={"": 0},\n                                                        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=dtype))\npipe = QwenImageEditPlusPipeline.from_pretrained(MODEL, transformer=None, vae=None, text_encoder=te, torch_dtype=dtype)\nemb = {}\nfor p in todo:\n    img = Image.open(p).convert("RGB").resize((R, R), Image.LANCZOS)\n    cw, ch = m.calculate_dimensions(m.CONDITION_IMAGE_SIZE, img.width / img.height)\n    cond = [pipe.image_processor.resize(img, ch, cw)]  # same condition image the pipeline builds itself\n    with torch.no_grad():\n        pe, pm = pipe.encode_prompt(prompt=FRONT, image=cond, device=dev)\n        ne, nm = pipe.encode_prompt(prompt=NEG, image=cond, device=dev)\n    assert torch.isfinite(pe).all() and torch.isfinite(ne).all(), "non-finite text embeddings (try --dtype bf16)"\n    emb[p.name] = tuple(None if x is None else x.cpu() for x in (pe, pm, ne, nm))  # the mask is None when it would be all ones\ndel pipe, te; free()\n\n# 2) GGUF transformer + VAE only\npath = hf_hub_download(GGUF_REPO, args.gguf)\ntr = QwenImageTransformer2DModel.from_single_file(path, quantization_config=GGUFQuantizationConfig(compute_dtype=dtype), config=MODEL,\n                                                  subfolder="transformer", torch_dtype=dtype).to(dev)\npipe = QwenImageEditPlusPipeline.from_pretrained(MODEL, transformer=tr, text_encoder=None, tokenizer=None, processor=None, torch_dtype=dtype).to(dev)\npipe.set_progress_bar_config(disable=True)\nfcntl.flock(load_lock, fcntl.LOCK_UN)\ntimings = []\nfor k, p in enumerate(todo, 1):\n    img = Image.open(p).convert("RGB").resize((R, R), Image.LANCZOS)\n    pe, pm, ne, nm = (None if x is None else x.to(dev) for x in emb[p.name])\n    torch.cuda.synchronize(); t0 = time.time()\n    with torch.inference_mode():\n        res = pipe(image=[img], prompt_embeds=pe.to(dtype), prompt_embeds_mask=pm, negative_prompt_embeds=ne.to(dtype), negative_prompt_embeds_mask=nm,\n                   true_cfg_scale=args.cfg, height=R, width=R, num_inference_steps=args.steps, generator=torch.Generator(dev).manual_seed(args.seed))\n    im = res.images[0]\n    torch.cuda.synchronize(); timings.append(time.time() - t0)\n    assert np.asarray(im).max() > 0 and np.asarray(im).std() > 1, f"black / flat output for {p.name} (fp16 overflow? try --dtype bf16)"\n    im.save(out / p.name)\n    print(f"{p.name}: {timings[-1]:.0f}s ({k}/{len(todo)})", flush=True)\n(out / f"params_shard{shard_k}.json").write_text(json.dumps({"model": MODEL, "gguf": f"{GGUF_REPO}/{args.gguf}", "prompt": FRONT, "dtype": args.dtype, "steps": args.steps,\n                                             "true_cfg_scale": args.cfg, "res": R, "seed": args.seed, "mean_seconds": float(np.mean(timings))}, indent=2))\nprint("done")\n')

In [ ]:
import subprocess, sys
P = os.path.dirname([f for f in glob.glob('/kaggle/input/**/0001.png', recursive=True) if 'real-portraits' in f][0]); print('photos:', P)
O = '/kaggle/working/qwen_front'
NG = torch.cuda.device_count(); print('GPUs:', NG)
def launch(shards):
    # fp16 gave NaN / black images and Q4_K_M + VAE ran out of memory, so Q3_K_M in bf16; ~13 min per image on a T4, hence one process per GPU
    ps = []
    for s in shards:
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(s % NG))
        ps.append((s, subprocess.Popen([sys.executable, '-u', '/kaggle/temp/code/qwen_front.py', '--input', P, '--output', O, '--limit', '0',
                                        '--gguf', 'Qwen-Image-Edit-2509-Q3_K_M.gguf', '--dtype', 'bf16', '--shard', '%d/%d' % (s, NG)], env=env)))
    return [s for s, p in ps if p.wait() != 0]
failed = launch(range(NG))
if failed:
    print('shards failed:', failed, 'retrying once (resumable)', flush=True)
    failed = launch(failed)
assert not failed, 'Qwen shards failed, see the log above'
assert len(glob.glob(O + '/*.png')) == (0 or 50)